# Post-processing: the candidates' final predictions

This notebook turns the setups' test predictions on the **final window** (trained by
`train_final-window.py`, one subfolder per setup in this folder) into each candidate's
final predictions, `final-predictions-<candidate>.parquet` (columns Date, Code, label,
Prediction), plus `post-processing.json` (weights, rows, dates).

The rules were fixed on 2026-11-05, before any final-window result existed
(`claude/project-notes.md`, T-20):

1. **Per-day demeaning** (each day's mean prediction across stocks is subtracted) for
   every candidate except **Run-2**, the author's baseline, which is scored raw as
   intended.
2. **Mixes** average their members' demeaned predictions on the rows all members share.
   - *Equal weights*: 1/k each.
   - *Fitted weights*: non-negative, summing to 1, minimizing the squared error of the
     mix on the final window's **out-of-fold validation predictions**
     (`predictions/validation_*.parquet`): each history row predicted by the fold model
     that did not train on it, at the tree count the final model uses. History only, so
     no test information enters the weights. Demeaned per day like the test predictions.
3. Nothing else: no rescaling, clipping or shrinkage.

Run it after `train_final-window.py` has finished every setup.

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import polars as pl
from scipy.optimize import minimize

# The notebook lives in runs/run-final/; TS48_FINAL_DIR overrides it (tests).
FINAL_DIR = Path(os.environ.get("TS48_FINAL_DIR", Path.cwd())).resolve()
SETUPS = ["run-2", "run-5", "run-11", "run-13", "run-14", "run-15", "run-16", "run-17",
          "run-22", "run-23"]
CANDIDATES = {  # name -> (members, demeaned per day, weighting); order = table order
    "run-2": (["run-2"], False, "single"),
    "run-5": (["run-5"], True, "single"),
    "run-22": (["run-22"], True, "single"),
    "run-23": (["run-23"], True, "single"),
    "run-11": (["run-11"], True, "single"),
    "run-17": (["run-17"], True, "single"),
    "mix-11-17-equal": (["run-11", "run-17"], True, "equal"),
    "mix-11-17-fitted": (["run-11", "run-17"], True, "fitted"),
    "lagmix-equal": (["run-11", "run-13", "run-14", "run-15", "run-16"], True, "equal"),
    "lagmix-fitted": (["run-11", "run-13", "run-14", "run-15", "run-16"], True, "fitted"),
}

missing = [s for s in SETUPS if not (FINAL_DIR / s / "done.json").exists()]
assert not missing, f"Setups not finished (no done.json): {missing}"


def final_window(setup: str) -> tuple[str, str]:
    """Test start and end of the setup's last (final) window."""
    results = json.loads((FINAL_DIR / setup / "results_lgbm.json").read_text())
    last = max(results, key=lambda r: r["test_start"])
    return last["test_start"], last["test_end"]


windows = {s: final_window(s) for s in SETUPS}
assert len(set(windows.values())) == 1, f"Setups disagree on the final window: {windows}"
TEST_START, TEST_END = windows["run-22"]
print(f"Final window: {TEST_START} -> {TEST_END}; folder {FINAL_DIR}")


def window_file(setup: str, kind: str) -> Path:
    """Return the setup's final-window file of a kind ("predictions" or "validation")."""
    files = [p for p in (FINAL_DIR / setup).rglob(f"{kind}_*.parquet")
             if f"_test_{TEST_START}_" in p.name]
    assert len(files) == 1, (setup, kind, files)
    return files[0]


def demean(frame: pl.DataFrame) -> pl.DataFrame:
    """Subtract each day's mean prediction across stocks."""
    return frame.with_columns(pl.col("Prediction") - pl.col("Prediction").mean().over("Date"))


def combine(frames: list[pl.DataFrame], weights: np.ndarray) -> pl.DataFrame:
    """Weighted sum of the members' predictions on the rows they all share."""
    out = frames[0].select("Date", "Code", "label",
                           (pl.col("Prediction") * weights[0]).alias("Prediction"))
    for frame, w in zip(frames[1:], weights[1:], strict=True):
        out = (out.join(frame.select("Date", "Code", pl.col("Prediction").alias("_p")),
                        on=["Date", "Code"])
               .with_columns(pl.col("Prediction") + w * pl.col("_p")).drop("_p"))
    return out.sort("Date", "Code")

## 1. The setups' test predictions on the final window

Rows per setup. The family setups share one universe (raw features and label finite);
Run-2's own feature filter can keep a slightly different set of rows. The evaluation
notebook scores every candidate on the rows they all share.

In [ ]:
test = {}
for s in SETUPS:
    test[s] = pl.read_parquet(window_file(s, "predictions")).select(
        "Date", "Code", "label", "Prediction")
    df = test[s]
    print(f"{s:7} {df.height:>9,} rows, {df['Date'].n_unique()} dates, "
          f"prediction mean {df['Prediction'].mean():+.5f}, SD {df['Prediction'].std():.5f}")

## 2. Fitted weights (out-of-fold validation predictions, history only)

For each mix: the weights that minimize the squared error of the demeaned mix on the
final window's out-of-fold predictions, under w ≥ 0 and Σw = 1. Shown with each member's
and each mix's out-of-fold Rank IC and R², for reference: these are in-sample for the
fitted weights, so they flatter them.

Caveats noted when the candidates were chosen (T-17): the out-of-fold predictions come
from fold models trained on 80% of the history, and validation favoured the members
closest to lag 1 (N-37).

In [ ]:
def fit_weights(frames: list[pl.DataFrame]) -> tuple[np.ndarray, int]:
    """Weights >= 0 summing to 1 that minimize the mix's squared error; rows used."""
    joined = frames[0].select("Date", "Code", "label", pl.col("Prediction").alias("p0"))
    for k, frame in enumerate(frames[1:], start=1):
        joined = joined.join(frame.select("Date", "Code", pl.col("Prediction").alias(f"p{k}")),
                             on=["Date", "Code"])
    p = joined.select([f"p{k}" for k in range(len(frames))]).to_numpy().astype(float)
    y = joined["label"].to_numpy().astype(float)
    a, b, k = p.T @ p / len(y), p.T @ y / len(y), len(frames)
    scale = np.trace(a) / k  # objective of order 1, so the optimizer's tolerance bites
    result = minimize(
        lambda w: (w @ a @ w - 2 * b @ w) / scale, np.full(k, 1 / k),
        jac=lambda w: (2 * a @ w - 2 * b) / scale, method="SLSQP", bounds=[(0, 1)] * k,
        constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1, "jac": lambda _w: np.ones(k)}],
        options={"ftol": 1e-15, "maxiter": 1000},
    )
    assert result.success, result.message
    w = np.clip(result.x, 0, None)
    return w / w.sum(), joined.height


def scores(frame: pl.DataFrame) -> tuple[float, float]:
    """Mean daily Rank IC and pooled R² of a (demeaned) prediction frame."""
    ric = frame.group_by("Date").agg(pl.corr(pl.col("Prediction").rank(), pl.col("label").rank()))
    y, p = frame["label"].to_numpy(), frame["Prediction"].to_numpy()
    return float(ric["Prediction"].mean()), float(1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum())


oof = {}
fitted = {}
for name, (members, _, how) in CANDIDATES.items():
    if how != "fitted":
        continue
    for m in members:
        oof.setdefault(m, demean(pl.read_parquet(window_file(m, "validation")).select(
            "Date", "Code", "label", "Prediction")))
    frames = [oof[m] for m in members]
    w, rows = fit_weights(frames)
    equal = np.full(len(members), 1 / len(members))
    fitted[name] = {"weights": w, "rows": rows,
                    "dates": [str(oof[members[0]]["Date"].min()), str(oof[members[0]]["Date"].max())]}
    print(f"{name}: fitted on {rows:,} out-of-fold rows "
          f"({fitted[name]['dates'][0]} -> {fitted[name]['dates'][1]})")
    for m, wm in zip(members, w, strict=True):
        ic, r2 = scores(oof[m])
        print(f"   {m:7} weight {wm:.3f} | out-of-fold Rank IC {ic:.4f}, R² {r2:.5f}")
    for label, ww in (("equal", equal), ("fitted", w)):
        ic, r2 = scores(combine(frames, ww))
        print(f"   mix, {label:6} weights: out-of-fold Rank IC {ic:.4f}, R² {r2:.5f}")

## 3. The candidates' final predictions

Each candidate's final predictions on the test rows of the final window, saved as
`final-predictions-<candidate>.parquet`.

In [ ]:
summary = {"test_start": TEST_START, "test_end": TEST_END, "candidates": {}}
for name, (members, demeaned, how) in CANDIDATES.items():
    frames = [demean(test[m]) if demeaned else test[m] for m in members]
    if how == "single":
        w = np.array([1.0])
    elif how == "equal":
        w = np.full(len(members), 1 / len(members))
    else:
        w = fitted[name]["weights"]
    final = combine(frames, w)
    path = FINAL_DIR / f"final-predictions-{name}.parquet"
    final.write_parquet(path)
    summary["candidates"][name] = {
        "members": members, "demeaned": demeaned, "weighting": how,
        "weights": [round(float(x), 6) for x in w], "rows": final.height,
        **({"fitted_on": {k: v for k, v in fitted[name].items() if k != "weights"}}
           if how == "fitted" else {}),
    }
    print(f"{name:17} {final.height:>9,} rows  weights "
          + " ".join(f"{m}={x:.3f}" for m, x in zip(members, w, strict=True)) + f"  -> {path.name}")
(FINAL_DIR / "post-processing.json").write_text(json.dumps(summary, indent=2))
print("wrote post-processing.json")